# 01. Auditoria e Qualidade de Dados (Data Audit)

## Objetivos da Auditoria
1. Validar volumetria e tipos de dados brutos.
2. Identificar percentual de nulos e duplicatas.
3. Checar a data exata de corte da base (`date_added`).
4. Validar premissas de integridade temporal (ex: checar inconsistências para H7).

In [4]:
import pandas as pd
import numpy as np 

df_raw = pd.read_csv(r'C:\Users\lukin\OneDrive\Documentos\netflix_catalog_analysis\data\netflix_titles.csv')

print(f"Dimensão da Base: {df_raw.shape[0]} linhas e {df_raw.shape[1]} colunas")
df_raw.head()

Dimensão da Base: 8807 linhas e 12 colunas


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [5]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8807 non-null   object
 1   type          8807 non-null   object
 2   title         8807 non-null   object
 3   director      6173 non-null   object
 4   cast          7982 non-null   object
 5   country       7976 non-null   object
 6   date_added    8797 non-null   object
 7   release_year  8807 non-null   int64 
 8   rating        8803 non-null   object
 9   duration      8804 non-null   object
 10  listed_in     8807 non-null   object
 11  description   8807 non-null   object
dtypes: int64(1), object(11)
memory usage: 825.8+ KB


In [9]:
null_summary = pd.DataFrame({
    'Total_Nulos': df_raw.isnull().sum(),
    'Percentual_Nulos (%)': (df_raw.isnull().sum() / len(df_raw)) * 100
}).sort_values(by='Percentual_Nulos (%)', ascending=False)

print("--- Resumo de Valores Ausentes ---")
display(null_summary)

print(f"\nTotal de linhas duplicadas (absolutas): {df_raw.duplicated().sum()}")
print(f"Total de IDs unicos (show_id): {df_raw['show_id'].nunique()}")

--- Resumo de Valores Ausentes ---


,Total_Nulos,Percentual_Nulos (%)
director,2634,29.908028
country,831,9.435676
cast,825,9.367549
date_added,10,0.113546
rating,4,0.045418
duration,3,0.034064
show_id,0,0.000000
type,0,0.000000
title,0,0.000000
release_year,0,0.000000



Total de linhas duplicadas (absolutas): 0
Total de IDs unicos (show_id): 8807


In [ ]:
df_audit = df_raw.copy()
df_audit['date_added_clean'] = pd.to_datetime(df_audit['date_added'].str.strip(), errors='coerce')

min_date = df_audit['date_added_clean'].min()
max_date = df_audit['date_added_clean'].max()

print(f"Data inicial no catálogo: {min_date.strftime('%Y-%m-%d') if pd.notnull(min_date) else 'N/A'}")
print(f"Data limite de corte (Máxima): {max_date.strftime('%Y-%m-%d') if pd.notnull(max_date) else 'N/A'}")

df_audit['release_year'] = pd.to_numeric(df_audit['release_year'], errors='coerce')
inconsistent_dates = df_audit[df_audit['date_added_clean'].dt.year < df_audit['release_year']]

print(f"\nInconsistências para H7 (date_added < release_year): {len(inconsistent_dates)} registros encontrados.")
if len(inconsistent_dates) > 0:
    display(inconsistent_dates[['show_id', 'title', 'release_year', 'date_added']])

Data inicial no catálogo: 2008-01-01
Data limite de corte (Máxima): 2021-09-25

Inconsistências para H7 (date_added < release_year): 14 registros encontrados.


,show_id,title,release_year,date_added
1551,s1552,Hilda,2021,"December 14, 2020"
1696,s1697,Polly Pocket,2021,"November 15, 2020"
2920,s2921,Love Is Blind,2021,"February 13, 2020"
3168,s3169,Fuller House,2020,"December 6, 2019"
3287,s3288,Maradona in Mexico,2020,"November 13, 2019"
3369,s3370,BoJack Horseman,2020,"October 25, 2019"
3433,s3434,The Hook Up Plan,2020,"October 11, 2019"
4844,s4845,Unbreakable Kimmy Schmidt,2019,"May 30, 2018"
4845,s4846,Arrested Development,2019,"May 29, 2018"
5394,s5395,Hans Teeuwen: Real Rancour,2018,"July 1, 2017"


In [15]:
print("--- Cardinalidade Inicial ---")
print(f"Tipos de midia unicos: {df_audit['type'].unique()}")
print(f"Ratings unicos: {df_audit['rating'].unique()}")
print(f"Paises unicos: (sem explode): {df_audit['country'].nunique()}")
print(f"Generos unicos (sem explode): {df_audit['listed_in'].nunique()}")

--- Cardinalidade Inicial ---
Tipos de midia unicos: ['Movie' 'TV Show']
Ratings unicos: ['PG-13' 'TV-MA' 'PG' 'TV-14' 'TV-PG' 'TV-Y' 'TV-Y7' 'R' 'TV-G' 'G'
 'NC-17' '74 min' '84 min' '66 min' 'NR' nan 'TV-Y7-FV' 'UR']
Paises unicos: (sem explode): 748
Generos unicos (sem explode): 514


In [20]:
rating_contamination = df_raw[df_raw['rating'].str.contains('min', na=False)]
print("--- Linhas com Deslocamento de Coluna (rating contendo min) ---")
display(rating_contamination[['show_id', 'title', 'rating', 'duration']])

print("\n--- Validação de Chave Primária e Duplicatas ---")
print(f"Total de linhas na base: {len(df_raw)}")
print(f"Total de IDs unicos (show_id): {df_raw['show_id'].nunique()}")
print(f"Total de linhas 100% duplicadas: {df_raw.duplicated().sum()}")

--- Linhas com Deslocamento de Coluna (rating contendo min) ---


,show_id,title,rating,duration
5541,s5542,Louis C.K. 2017,74 min,NaN
5794,s5795,Louis C.K.: Hilarious,84 min,NaN
5813,s5814,Louis C.K.: Live at the Comedy Store,66 min,NaN



--- Validação de Chave Primária e Duplicatas ---
Total de linhas na base: 8807
Total de IDs unicos (show_id): 8807
Total de linhas 100% duplicadas: 0


## 6. Síntese da Auditoria & Regras de Negócio para Limpeza (Data Cleaning Pipeline)

Após a execução da auditoria sobre os 8.807 registros brutos, foram mapeados os seguintes comportamentos e decisões metodológicas:

### 1. Deslocamento de Coluna (Bug Conhecido da Base)
- **Achado:** 3 registros possuem a duração em minutos gravada na coluna `rating` (`74 min`, `84 min`, `66 min`), resultando em nulos na coluna `duration`.
- **Regra no ETL:** Mover o valor de `rating` para `duration` nestas 3 linhas e atribuir `rating = 'NR'` (Not Rated / Não Avaliado).

### 2. Tratamento de Inconsistências Temporais (H7 - Lag Negativo)
- **Achado:** 14 registros (0,16% do catálogo) possuem `date_added.year < release_year` (ex: *Fuller House*, *BoJack Horseman*), decorrente da atualização do ano de lançamento de novas temporadas de séries.
- **Regra no ETL:** Manter as linhas no dataset geral, mas aplicar um filtro excludente especificamente na métrica de cálculo de *lag* da **H7** (`lag >= 0`).

### 3. Tratamento de Valores Nulos
- **`date_added` (10 nulos / 0,11%):** Descartar apenas durante o processamento das hipóteses temporais (H1, H2, H4, H5, H6, H7).
- **`country` (831 nulos / 9,43%):** Imputar com a string `'Unknown'` antes da etapa de *split/explode*.
- **`rating` (4 nulos / 0,04%):** Imputar com a classe `'NR'` (Not Rated).

### 4. Integridade da Base
- A coluna `show_id` é 100% única (8.807 registros), atuando como chave primária válida. Não foram detectadas duplicatas absolutas.
- Data de corte estática confirmada em **25 de Setembro de 2021**.

### 5. Decisões de Cálculo e Sanitização Fina (Refinamento Metodológico)
- **H2 (Expansão Internacional):** O valor `'Unknown'` atribuído a 831 registros (9,43%) será mantido para integridade da base, mas **estritamente excluído do numerador e do denominador** no cálculo do percentual de produções fora dos EUA. A métrica avaliará apenas títulos com origem conhecida.
- **Sanitização de Arrays em Explode:** Todo desmembramento por vírgula em `country` e `listed_in` aplicará `.str.strip()` para eliminar espaçamentos nas pontas (ex: evitar que `' France'` e `'France'` sejam contabilizados como países distintos).